In [11]:
import numpy as np
from scipy.sparse import identity
from pauli_ops import get_pauli

In [12]:
def jw_annihilation(p, num_modes):
    """ Operador de aniquilación fermiónica c_p bajo Jordan-Wigner,
    como matriz dispersa en el espacio de num_modes qubits."""
    # cadena Z sobre los modos 0..p-1
    string = identity(2**num_modes, format='csr', dtype=complex)
    for q in range(p):
        string = string @ get_pauli(q, num_modes, 'z')
    # sigma^- en el modo p
    sigma_minus = (get_pauli(p, num_modes, 'x') + 1j*get_pauli(p, num_modes, 'y')) / 2
    return string @ sigma_minus

In [13]:
def jw_creation(p, num_modes):
    return jw_annihilation(p, num_modes).getH()

def jw_number(p, num_modes):
    c = jw_annihilation(p, num_modes)
    return (c.getH() @ c).real   # n_p = c_p^dag c_p, es diagonal 0/1

In [15]:
def fock_state(occupied, num_modes):
    """Vector de la base con los modos en `occupied` ocupados.
    Ojo: el bit del modo q va en la posicion (num_modes-1-q) porque
    get_pauli pone el qubit 0 como el factor tensorial mas a la izquierda."""
    idx = sum(1 << (num_modes - 1 - q) for q in occupied)
    psi = np.zeros(2**num_modes, dtype=complex)
    psi[idx] = 1.0
    return psi

In [16]:
def expect(op, psi):
    return np.real(psi.conj() @ (op @ psi))

# ===================== TESTS =====================
def test_numero():
    num_modes = 4
    psi = fock_state([0, 2], num_modes)                 # ocupo modos 0 y 2
    obtenido = [round(expect(jw_number(p, num_modes), psi)) for p in range(num_modes)]
    assert obtenido == [1, 0, 1, 0], obtenido
    print('OK  test_numero')

def test_creacion():
    num_modes = 4
    p = 2
    vac = fock_state([], num_modes)                     # |vacío>
    obtenido = jw_creation(p, num_modes) @ vac          # c_p^dag |vacío>
    esperado = fock_state([p], num_modes)               # modo p ocupado
    # abs() para ignorar el signo de la cadena de Jordan-Wigner
    assert np.allclose(np.abs(obtenido), np.abs(esperado)), 'el estado no coincide'
    assert np.isclose(np.linalg.norm(obtenido), 1.0), 'no está normalizado'
    print('OK  test_creacion')

def test_doble_ocupacion():
    # convención intercalada: modo(sitio, espín) = 2*sitio + espín  (0=up, 1=down)
    L, num_modes = 2, 4                                  # 2 sitios -> 4 modos
    def modo(sitio, espin): return 2*sitio + espin
    n_up_dn = lambda s: jw_number(modo(s,0), num_modes) @ jw_number(modo(s,1), num_modes)

    # sitio 0 doblemente ocupado (up y down), sitio 1 vacío
    psi = fock_state([modo(0,0), modo(0,1)], num_modes)
    assert np.isclose(expect(n_up_dn(0), psi), 1.0), 'sitio 0 debería estar doble-ocupado'
    assert np.isclose(expect(n_up_dn(1), psi), 0.0), 'sitio 1 debería estar vacío'

    # sitio 0 con ocupación SIMPLE (solo up) -> doble ocupación = 0
    psi2 = fock_state([modo(0,0)], num_modes)
    assert np.isclose(expect(n_up_dn(0), psi2), 0.0), 'ocupación simple no cuenta como doble'
    print('OK  test_doble_ocupacion')

# ===================== CORRER =====================
test_numero()
test_creacion()
test_doble_ocupacion()
print('--- todos los tests pasaron ---')

OK  test_numero
OK  test_creacion
OK  test_doble_ocupacion
--- todos los tests pasaron ---


In [18]:
def build_fermi_hubbard(Lx, Ly, t, U, periodic=False):
    L = Lx*Ly
    num_modes = 2*L
    def modo(sitio, espin): return 2*sitio + espin
    def sitio(fila, col):   return fila*Lx + col

    # --- enlaces de la red 2D (set de tuplas ordenadas -> sin duplicados) ---
    enlaces = set()
    for fila in range(Ly):
        for col in range(Lx):
            s = sitio(fila, col)
            if col+1 < Lx:                       # vecino derecha
                enlaces.add(tuple(sorted((s, sitio(fila, col+1)))))
            elif periodic and Lx > 2:            # wrap horizontal
                enlaces.add(tuple(sorted((s, sitio(fila, 0)))))
            if fila+1 < Ly:                      # vecino abajo
                enlaces.add(tuple(sorted((s, sitio(fila+1, col)))))
            elif periodic and Ly > 2:            # wrap vertical
                enlaces.add(tuple(sorted((s, sitio(0, col)))))

    H = csr_matrix((2**num_modes, 2**num_modes), dtype=complex)

    # --- hopping:  -t (c_i^dag c_j + h.c.)  por enlace y por espín ---
    for (i, j) in enlaces:
        for espin in (0, 1):
            cdag_i = jw_creation(modo(i, espin), num_modes)
            c_j    = jw_annihilation(modo(j, espin), num_modes)
            hop = cdag_i @ c_j
            H = H - t*(hop + hop.getH())         # hop.getH() = c_j^dag c_i (el h.c.)

    # --- on-site:  U * n_up n_down  por sitio ---
    for s in range(L):
        n_up = jw_number(modo(s, 0), num_modes)
        n_dn = jw_number(modo(s, 1), num_modes)
        H = H + U*(n_up @ n_dn)

    return H
    